In [1]:
import pandas as pd

print("Loading dataset into memory...")
df = pd.read_excel('../data/ECU_IoHT.xlsx')

memory_usage = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Dataset loaded! It is using {memory_usage:.2f} MB of RAM.\n")

print("--- COLUMN NAMES ---")
print(df.columns.tolist())

print("\n--- FIRST ROW PREVIEW ---")
print(df.head(1))

Loading dataset into memory...
Dataset loaded! It is using 51.30 MB of RAM.

--- COLUMN NAMES ---
['No.', 'Time', 'Source', 'Destination', 'Protocol', 'Length', 'Info', 'Type', 'Type of attack']

--- FIRST ROW PREVIEW ---
   No.  Time         Source Destination Protocol  Length  \
0    1   0.0  Alfa_97:cf:63   Broadcast      ARP      42   

                                        Info    Type Type of attack  
0  Who has 192.168.43.1? Tell 192.168.43.186  Attack   ARP Spoofing  


In [ ]:
print("Cleaning data...")

df_clean = df[['Protocol', 'Info', 'Type']].copy()

df_clean = df_clean.dropna()

df_clean['bert_text'] = df_clean['Protocol'].astype(str) + " " + df_clean['Info'].astype(str)

print("Label types found:", df_clean['Type'].unique())

df_clean['label'] = df_clean['Type'].apply(lambda x: 1 if 'attack' in str(x).lower() else 0)

df_final = df_clean[['bert_text', 'label']]

print("\n--- CLEANED DATA PREVIEW ---")
print(df_final.head())

print("\n--- ATTACK VS BENIGN COUNT ---")
print(df_final['label'].value_counts())

Cleaning data...
Label types found: ['Attack' 'Normal']

--- CLEANED DATA PREVIEW ---
                                           bert_text  label
0      ARP Who has 192.168.43.1? Tell 192.168.43.186      1
1           ARP 192.168.43.1 is at 6e:c7:ec:3c:f2:ba      1
2      ARP Who has 192.168.43.1? Tell 192.168.43.186      1
3  DNS Standard query 0x0c44 PTR 1.43.168.192.in-...      0
4           ARP 192.168.43.1 is at 6e:c7:ec:3c:f2:ba      1

--- ATTACK VS BENIGN COUNT ---
label
1    87754
0    23453
Name: count, dtype: int64


In [ ]:
df_majority = df_final[df_final['label'] == 1]
df_minority = df_final[df_final['label'] == 0]

df_majority_downsampled = df_majority.sample(n=len(df_minority), random_state=42)

df_balanced = pd.concat([df_majority_downsampled, df_minority])


df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("--- NEW BALANCED COUNT ---")
print(df_balanced['label'].value_counts())

clean_file_path = '../data/ecu_ioht_clean.csv'
df_balanced.to_csv(clean_file_path, index=False)
print(f"\nClean data saved to: {clean_file_path}")

--- NEW BALANCED COUNT ---
label
1    23453
0    23453
Name: count, dtype: int64

Clean data saved to: ../data/ecu_ioht_clean.csv
